In [2]:
import pandas as pd
import os
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch


from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [4]:
data = pd.read_csv('lemmatized_and_prepared_df_2810.csv', sep = ',')

In [5]:
data = data[data['salary_from'] >= 1000]

In [6]:
data

,Unnamed: 0,salary_from,experience_from,experience_to,no_experience,city_воронеж,city_деревня,city_екатеринбур,city_ижевск,city_казань,...,title_key_электросварщик,title_key_энергетик,title_key_юрисконсульт,title_key_юрист,month_4,month_5,month_6,month_8,month_9,description_lemmatized
0,0,160000.0,6.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,вакансия компания ооо пк предприятие пик предп...
1,1,62000.0,1.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,компания зао росм один из лидер российский рын...
2,2,130000.0,1.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,гк миланстрой требоваться начальник участок фа...
3,3,80000.0,1.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,гк олимпроект лидировать компания сфера проект...
4,4,250000.0,6.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,компания ортоника являться один из ведущий рос...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
332049,332049,200000.0,3.0,6.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,requirement подтвердить навык построение систе...
332050,332050,350000.0,1.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,requirement опыт работа spring boot spring sec...
332051,332051,200000.0,3.0,6.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,requirement опыт работа на проект по внедрение...
332052,332052,90000.0,1.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,requirement высокий профильный образование тео...


In [7]:
data = data.sample(frac=0.5)

In [8]:
data

,Unnamed: 0,salary_from,experience_from,experience_to,no_experience,city_воронеж,city_деревня,city_екатеринбур,city_ижевск,city_казань,...,title_key_электросварщик,title_key_энергетик,title_key_юрисконсульт,title_key_юрист,month_4,month_5,month_6,month_8,month_9,description_lemmatized
118588,118588,50000.0,1.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,обязанность организовать деятельность смена ск...
192933,192933,60000.0,1.0,3.0,True,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,лучше звонить клининговая компания это професс...
323311,323311,120000.0,1.0,6.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,обязанность встречать приветствовать клиент вз...
149306,149306,35000.0,1.0,3.0,True,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,обязанность продвижение развитие работа графич...
6785,6785,135780.0,1.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,обязанность обеспечение эффективный уход за тр...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32056,32056,61000.0,1.0,6.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,розничный сеть красный белый требоваться специ...
279310,279310,45000.0,1.0,3.0,False,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,обязанность фс маркер игрушка приглашать на ра...
300348,300348,35000.0,6.0,3.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,обязанность качественный пошив изделие для реб...
193397,193397,55000.0,1.0,6.0,False,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,обязанность поиск поставщик оборудование матер...


In [9]:
data.drop('Unnamed: 0', axis = 1, inplace = True)

In [10]:
y = data["salary_from"][data.index]
data = data.loc[data["salary_from"].notnull(), :].drop(
    "salary_from", axis=1
)

In [11]:
y_log = np.log1p(y)

In [74]:
y_log

,salary_from
58250,10.150387
319808,10.819798
12656,11.775297
158652,9.903538
18251,11.082158
...,...
247744,11.289794
121347,11.314487
89070,10.463132
234662,10.714440


In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from scipy.sparse import hstack

import nltk
nltk.download('stopwords')  # если еще не скачали
from nltk.corpus import stopwords
russian_stop_words = stopwords.words('russian')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# Сначала обработаем пропущенные значения
def preprocess_text_data(X_train, text_column):
    """Обработка пропущенных значений в текстовых данных"""

    # Заполняем NaN пустыми строками
    data_clean = data.copy()

    data_clean[text_column] = data_clean[text_column].fillna('')

    # Проверяем результат
    print(f"Пропуски после обработки: {data_clean[text_column].isna().sum()}")

    return data_clean

# Обрабатываем пропуски
data_clean = preprocess_text_data(data, 'description_lemmatized')

# Создаем TF-IDF векторaйзер
tfidf = TfidfVectorizer(
    max_features=1000,
    min_df=2,
    max_df=0.8,
    stop_words=russian_stop_words
)

# Применяем TF-IDF к ОЧИЩЕННЫМ данным
data_tfidf = tfidf.fit_transform(data_clean['description_lemmatized'])

Пропуски после обработки: 0


In [14]:
data_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10121205 stored elements and shape (165825, 1000)>

In [ ]:
# Объединяем BOW с другими признаками

data_other = data_clean.drop('description_lemmatized', axis=1)

numeric_cols = data_other.select_dtypes(include=['int64', 'float64']).columns
data_numeric = data_other[numeric_cols].values

data_tfidf_combined = hstack([data_numeric, data_tfidf])

In [ ]:
data_tfidf_combined = pd.DataFrame(data_tfidf_combined.toarray())

In [ ]:
class RobustDataFrameTransformer:
    def __init__(self, features_df, targets_series, clip_percentile=99):
        """
        features_df: DataFrame с фичами
        targets_series: Series с целевой переменной
        clip_percentile: процентиль для обрезки выбросов
        """
        # Сохраняем САМЫЕ ЧАСТОТНЫЕ ЗНАЧЕНИЯ (МОДЫ) для каждого признака
        self.features_mode_values = {}
        for col in features_df.columns:
            mode_values = features_df[col].mode()  # Находим самые частотные значения
            self.features_mode_values[col] = mode_values[0] if not mode_values.empty else features_df[col].median()

        # Сохраняем САМОЕ ЧАСТОТНОЕ ЗНАЧЕНИЕ для target
        target_mode = targets_series.mode()
        self.target_mode_value = target_mode[0] if not target_mode.empty else targets_series.median()

        # ЗАПОЛНЯЕМ NaN САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ
        features_filled, targets_filled = self._fill_na_with_mode(features_df, targets_series)

        # Конвертируем в numpy
        features_array = features_filled.values.astype(np.float32)
        targets_array = targets_filled.values.astype(np.float32)

        # ОБРЕЗАЕМ ВЫБРОСЫ перед вычислением статистик
        self.features_clip_value = np.percentile(np.abs(features_array), clip_percentile, axis=0)
        self.targets_clip_value = np.percentile(np.abs(targets_array), clip_percentile)

        features_clipped = np.clip(features_array, -self.features_clip_value, self.features_clip_value)
        targets_clipped = np.clip(targets_array, -self.targets_clip_value, self.targets_clip_value)

        # Вычисляем статистики на ОБРЕЗАННЫХ данных
        self.features_mean = np.mean(features_clipped, axis=0)
        self.features_std = np.std(features_clipped, axis=0)
        self.targets_mean = np.mean(targets_clipped)
        self.targets_std = np.std(targets_clipped)

        # Защита от деления на 0
        self.features_std = np.where(self.features_std < 1e-8, 1.0, self.features_std)
        if self.targets_std < 1e-8:
            self.targets_std = 1.0

        self.feature_columns = features_df.columns.tolist()
        self.clip_percentile = clip_percentile

        print(f"RobustDataFrameTransformer создан (clip {clip_percentile}%):")
        print(f"  Features: {len(self.feature_columns)} columns")
        print(f"  Targets: mean={self.targets_mean:.2f}, std={self.targets_std:.2f}")
        print(f"  Clip values - features: {np.max(self.features_clip_value):.2f}, targets: {self.targets_clip_value:.2f}")
        print(f"  NaN strategy: ЗАПОЛНЕНИЕ САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ")

    def _fill_na_with_mode(self, features_df, targets_series):
        """ЗАПОЛНЯЕТ NaN САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ"""

        # Проверяем наличие NaN
        features_nan_count = features_df.isna().sum().sum()
        targets_nan_count = targets_series.isna().sum()

        if features_nan_count > 0:
            print(f"Заполняем {features_nan_count} NaN в features САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ")
        if targets_nan_count > 0:
            print(f"Заполняем {targets_nan_count} NaN в targets САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ")

        # ЗАПОЛНЯЕМ FEATURES САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ
        features_filled = features_df.copy()
        for col, mode_value in self.features_mode_values.items():
            features_filled[col] = features_filled[col].fillna(mode_value)

        # ЗАПОЛНЯЕМ TARGET САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ
        targets_filled = targets_series.fillna(self.target_mode_value)

        return features_filled, targets_filled

    def transform_features(self, features_df):
        """Нормализует DataFrame с обрезкой выбросов и ЗАПОЛНЕНИЕМ NaN САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ"""
        # ЗАПОЛНЯЕМ NaN САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ
        features_filled = features_df.copy()
        for col, mode_value in self.features_mode_values.items():
            features_filled[col] = features_filled[col].fillna(mode_value)

        features_np = features_filled.values.astype(np.float32)
        features_clipped = np.clip(features_np, -self.features_clip_value, self.features_clip_value)
        features_norm = (features_clipped - self.features_mean) / self.features_std
        return features_norm

    def transform_targets(self, targets_series):
        """Нормализует Series с обрезкой выбросов и ЗАПОЛНЕНИЕМ NaN САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ"""
        # ЗАПОЛНЯЕМ NaN САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ
        targets_filled = targets_series.fillna(self.target_mode_value)

        targets_np = targets_filled.values.astype(np.float32)
        targets_clipped = np.clip(targets_np, -self.targets_clip_value, self.targets_clip_value)
        targets_norm = (targets_clipped - self.targets_mean) / self.targets_std
        return targets_norm

    def inverse_transform_targets(self, targets_norm):
        """Денормализует предсказания"""
        if isinstance(targets_norm, torch.Tensor):
            targets_norm = targets_norm.detach().cpu().numpy()
        return targets_norm * self.targets_std + self.targets_mean

In [ ]:
print("=== ПОДГОТОВКА ДАННЫХ ИЗ DATAFRAME ===")

# Шаг 1: Проверяем данные
print("Исходные данные:")
print(f"Features shape: {data_tfidf_combined.shape}")
print(f"Targets shape: {y_log.shape}")

print("\n=== СТАТИСТИКА ДО НОРМАЛИЗАЦИИ ===")
print("Features статистика:")
print(f"  Среднее: {data_tfidf_combined.mean().mean():.3f}")
print(f"  Std: {data_tfidf_combined.std().mean():.3f}")
print(f"  Min: {data_tfidf_combined.min().min():.3f}")
print(f"  Max: {data_tfidf_combined.max().max():.3f}")

print("Targets статистика:")
print(f"  Среднее: {y_log.mean():.3f}")
print(f"  Std: {y_log.std():.3f}")
print(f"  Min: {y_log.min():.3f}")
print(f"  Max: {y_log.max():.3f}")

# Шаг 2: Создаем трансформер на ВСЕХ данных
print("\n=== СОЗДАНИЕ ТРАНСФОРМЕРА ===")
transformer = RobustDataFrameTransformer(data_tfidf_combined, y_log)

# Шаг 3: Нормализуем данные
print("\n=== НОРМАЛИЗАЦИЯ ДАННЫХ ===")
normalized_features = transformer.transform_features(data_tfidf_combined)
normalized_targets = transformer.transform_targets(y_log)

print(f"Нормализованные features: {normalized_features.min():.3f} to {normalized_features.max():.3f}")
print(f"Нормализованные targets: {normalized_targets.min():.3f} to {normalized_targets.max():.3f}")

# Шаг 4: Преобразуем в тензоры и создаем DataLoader
print("\n=== СОЗДАНИЕ DATALOADER ===")
# Конвертируем в тензоры PyTorch
features_tensor = torch.FloatTensor(normalized_features)
targets_tensor = torch.FloatTensor(normalized_targets)

# Создаем Dataset
dataset = TensorDataset(features_tensor, targets_tensor)

# Разделяем на train/val
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Создаем DataLoader
batch_size = 64
train_hh_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_hh_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print("✅ DataLoader созданы из DataFrame!")

=== ПОДГОТОВКА ДАННЫХ ИЗ DATAFRAME ===
Исходные данные:
Features shape: (165825, 1233)
Targets shape: (165825,)

=== СТАТИСТИКА ДО НОРМАЛИЗАЦИИ ===
Features статистика:
  Среднее: 0.014
  Std: 0.040
  Min: 0.000
  Max: 6.000
Targets статистика:
  Среднее: 10.950
  Std: 0.595
  Min: 6.909
  Max: 14.221

=== СОЗДАНИЕ ТРАНСФОРМЕРА ===
RobustDataFrameTransformer создан (clip 99%):
  Features: 1233 columns
  Targets: mean=10.95, std=0.59
  Clip values - features: 6.00, targets: 12.43
  NaN strategy: ЗАПОЛНЕНИЕ САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ

=== НОРМАЛИЗАЦИЯ ДАННЫХ ===
Нормализованные features: -4.102 to 9.897
Нормализованные targets: -6.865 to 2.519

=== СОЗДАНИЕ DATALOADER ===
Train samples: 132660
Val samples: 33165
✅ DataLoader созданы из DataFrame!


In [20]:
def train_epoch(
    network,
    train_loader,
    criterion,
    optimizer,
    transformer=None
):
    network.train()
    total_loss = 0

    for feats, labels in train_loader:
        labels = labels.float()
        feats, labels = feats.to(device), labels.to(device)

        optimizer.zero_grad()

        logits = network(feats)
        labels = labels.squeeze()
        logits = logits.squeeze()

        if transformer is not None:
            # Вычисляем лосс в нормализованной шкале (сохраняем градиенты)
            loss = criterion(logits, labels)
        else:
            loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    mean_loss = total_loss / len(train_loader)
    print(f'Mean Train Loss: {mean_loss:.6f}')
    return total_loss / len(train_loader)

In [21]:
@torch.no_grad()
def val_epoch(
    network,
    val_loader,
    criterion,
    transformer=None
):
    val_loss = 0
    all_predictions_final = []
    all_targets_final = []
    all_predictions_norm = []  # для лосса в нормализованной шкале
    all_targets_norm = []      # для лосса в нормализованной шкале

    network.eval()
    for feats, labels in val_loader:
        labels = labels.float()
        feats, labels = feats.to(device), labels.to(device)

        logits = network(feats)
        logits = logits.squeeze()
        labels = labels.squeeze()

        # Сохраняем для лосса (в нормализованной шкале)
        all_predictions_norm.extend(logits.cpu().numpy())
        all_targets_norm.extend(labels.cpu().numpy())

        # Обратные преобразования только для метрик
        if transformer is not None:
            try:
                # 1. Обратное преобразование нормализации
                predictions_original = transformer.inverse_transform_targets(logits.cpu().numpy())
                targets_original = transformer.inverse_transform_targets(labels.cpu().numpy())

                # 2. Обратное преобразование log1p
                predictions_final = np.expm1(predictions_original)
                targets_final = np.expm1(targets_original)

                all_predictions_final.extend(predictions_final)
                all_targets_final.extend(targets_final)

            except:
                # Если нет методов inverse_transform, используем ручное преобразование
                if hasattr(transformer, 'target_mean_') and hasattr(transformer, 'target_std_'):
                    # Ручное обратное преобразование нормализации
                    predictions_original = logits.cpu().numpy() * transformer.target_std_ + transformer.target_mean_
                    targets_original = labels.cpu().numpy() * transformer.target_std_ + transformer.target_mean_

                    predictions_final = np.expm1(predictions_original)
                    targets_final = np.expm1(targets_original)

                    all_predictions_final.extend(predictions_final)
                    all_targets_final.extend(targets_final)
                else:
                    # Если не можем преобразовать, используем нормализованные значения
                    all_predictions_final.extend(logits.cpu().numpy())
                    all_targets_final.extend(labels.cpu().numpy())
        else:
            all_predictions_final.extend(logits.cpu().numpy())
            all_targets_final.extend(labels.cpu().numpy())

        # Лосс считаем в нормализованной шкале (чтобы был сопоставим с train)
        loss = criterion(logits, labels)
        val_loss += loss.item()

    # Метрики в исходной шкале если смогли преобразовать
    all_predictions_final = np.array(all_predictions_final)
    all_targets_final = np.array(all_targets_final)

    r2 = r2_score(all_targets_final, all_predictions_final)
    rmse = np.sqrt(np.mean((all_targets_final - all_predictions_final) ** 2))
    mean_val_loss = val_loss / len(val_loader)

    print(f'Val Loss: {mean_val_loss:.6f}, R²: {r2:.4f}, RMSE: {rmse:.2f} тыс. руб.')

    # Дополнительная информация
    if transformer is not None and len(all_predictions_final) > 0:
        print(f'Диапазон предсказаний: {all_predictions_final.min():.2f} - {all_predictions_final.max():.2f} тыс. руб.')
        print(f'Диапазон истинных значений: {all_targets_final.min():.2f} - {all_targets_final.max():.2f} тыс. руб.')

    return mean_val_loss, r2, rmse

In [22]:
def train_val(
    network,
    n_epochs,
    criterion,
    optimizer,
    train_loader,
    val_loader,
    transformer=None
):
    for epoch in range(1, n_epochs + 1):
        print(f'Epoch {epoch}/{n_epochs}')

        train_loss = train_epoch(network, train_loader, criterion, optimizer, transformer=transformer)
        val_loss, r2, rmse = val_epoch(network, val_loader, criterion, transformer=transformer)

        print('-' * 50)

    return val_loss, r2, rmse

In [23]:
import torch.nn as nn

class SimpleRegression(nn.Module):
    def __init__(self, input_size=1233):
        super().__init__()
        # Всего 1 выходной нейрон для предсказания одного числа
        self.linear = nn.Linear(input_size, 1)

    def forward(self, x):
        x = self.linear(x)

        return x.squeeze(1)

In [ ]:
model = SimpleRegression().to(device)

In [ ]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
train_val(model, 10, criterion, optimizer, train_hh_loader, val_hh_loader, transformer=transformer)

Epoch 1/10
Mean Train Loss: 0.526769
Val Loss: 0.518880, R²: 0.4529, RMSE: 30659.91 тыс. руб.
Диапазон предсказаний: 9805.13 - 430475.75 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 2/10
Mean Train Loss: 0.512779
Val Loss: 0.517144, R²: 0.4529, RMSE: 30659.51 тыс. руб.
Диапазон предсказаний: 9626.59 - 380632.12 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 3/10
Mean Train Loss: 0.511693
Val Loss: 0.516438, R²: 0.4506, RMSE: 30724.88 тыс. руб.
Диапазон предсказаний: 12886.14 - 365652.19 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 4/10
Mean Train Loss: 0.511577
Val Loss: 0.518543, R²: 0.4416, RMSE: 30975.95 тыс. руб.
Диапазон предсказаний: 10535.36 - 387059.69 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
-------------------------

(0.5175385554401861, 0.4523894786834717, np.float32(30674.092))

## Добавим 2 слоя (1233 -- 617 -- 1) через ReLu

In [ ]:
class TwoLayerRegression(nn.Module):
    def __init__(self, input_size=1233, hidden_size=617):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, 1)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)

        return x.squeeze(1)

In [ ]:
model_2 = TwoLayerRegression().to(device)

In [ ]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model_2.parameters(), lr=0.001)

In [ ]:
train_val(model_2, 10, criterion, optimizer, train_hh_loader, val_hh_loader, transformer=transformer)

Epoch 1/10
Mean Train Loss: 0.471649
Val Loss: 0.457644, R²: 0.5052, RMSE: 29158.84 тыс. руб.
Диапазон предсказаний: 4387.39 - 681445.12 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 2/10
Mean Train Loss: 0.404326
Val Loss: 0.416798, R²: 0.5626, RMSE: 27414.82 тыс. руб.
Диапазон предсказаний: 3530.03 - 492661.78 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 3/10
Mean Train Loss: 0.352607
Val Loss: 0.407194, R²: 0.5813, RMSE: 26822.42 тыс. руб.
Диапазон предсказаний: 2330.84 - 426715.78 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 4/10
Mean Train Loss: 0.303630
Val Loss: 0.412496, R²: 0.5845, RMSE: 26718.36 тыс. руб.
Диапазон предсказаний: 1336.14 - 471850.56 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
---------------------------

(0.4386713509385067, 0.5660426616668701, np.float32(27306.078))

## Добавим Dropout

In [ ]:
class TwoLayerRegressionDropout(nn.Module):
    def __init__(self, input_size=1233, hidden_size=617, dropout_rate=0.3):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, 1)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)

        return x.squeeze(1)

In [ ]:
model_3 = TwoLayerRegressionDropout().to(device)

In [ ]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model_3.parameters(), lr=0.001)

In [ ]:
train_val(model_3, 10, criterion, optimizer, train_hh_loader, val_hh_loader, transformer=transformer)

Epoch 1/10
Mean Train Loss: 0.484325
Val Loss: 0.458553, R²: 0.4996, RMSE: 29321.47 тыс. руб.
Диапазон предсказаний: 5846.03 - 294157.81 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 2/10
Mean Train Loss: 0.433675
Val Loss: 0.442894, R²: 0.5247, RMSE: 28576.64 тыс. руб.
Диапазон предсказаний: 3099.50 - 302626.03 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 3/10
Mean Train Loss: 0.401183
Val Loss: 0.417762, R²: 0.5591, RMSE: 27525.15 тыс. руб.
Диапазон предсказаний: 4238.36 - 413780.38 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 4/10
Mean Train Loss: 0.371225
Val Loss: 0.414198, R²: 0.5786, RMSE: 26907.74 тыс. руб.
Диапазон предсказаний: 1621.23 - 473352.78 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
---------------------------

(0.4084245197919982, 0.5969247817993164, np.float32(26316.545))

## Добавим еще два скрытых слоя (1233 - 1024 --  512 --  256)

In [ ]:
class FourLayerRegressionDropout(nn.Module):
    def __init__(self, input_size=1233, hidden_size1=1024,  hidden_size2=512,  hidden_size3=256, dropout_rate=0.3):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size1)
        self.linear2 = nn.Linear(hidden_size1, hidden_size2)
        self.linear3 = nn.Linear(hidden_size2, hidden_size3)
        self.linear4 = nn.Linear(hidden_size3, 1)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear3(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear4(x)

        return x.squeeze(1)

In [ ]:
model_5 = FourLayerRegressionDropout().to(device)

In [ ]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model_5.parameters(), lr=0.001)

In [ ]:
train_val(model_5, 10, criterion, optimizer, train_hh_loader, val_hh_loader, transformer=transformer)

Epoch 1/10
Mean Train Loss: 0.479616
Val Loss: 0.424259, R²: 0.5415, RMSE: 28067.33 тыс. руб.
Диапазон предсказаний: 6313.59 - 277312.16 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 2/10
Mean Train Loss: 0.415405
Val Loss: 0.418122, R²: 0.5955, RMSE: 26362.72 тыс. руб.
Диапазон предсказаний: 4559.69 - 451411.94 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 3/10
Mean Train Loss: 0.376045
Val Loss: 0.398785, R²: 0.5851, RMSE: 26700.91 тыс. руб.
Диапазон предсказаний: 3298.10 - 341053.50 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 4/10
Mean Train Loss: 0.342815
Val Loss: 0.403179, R²: 0.5822, RMSE: 26793.68 тыс. руб.
Диапазон предсказаний: 272.59 - 510109.97 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
----------------------------

(0.4182799554399443, 0.48931199312210083, np.float32(29621.951))

## Добавим нормализацию по батчу

In [ ]:
class FourLayerRegressionDropoutNorm(nn.Module):
    def __init__(self, input_size=1233, hidden_size1=1024,  hidden_size2=512,  hidden_size3=256, dropout_rate=0.3):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size1)
        self.linear2 = nn.Linear(hidden_size1, hidden_size2)
        self.linear3 = nn.Linear(hidden_size2, hidden_size3)
        self.linear4 = nn.Linear(hidden_size3, 1)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.bn1 = nn.BatchNorm1d(hidden_size1)
        self.bn1 = nn.BatchNorm1d(hidden_size2)
        self.bn1 = nn.BatchNorm1d(hidden_size3)

    def forward(self, x):
        x = self.linear1(x)
        x = self.bn1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        x = self.bn2(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear3(x)
        x = self.bn3(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear4(x)

        return x.squeeze(1)

In [ ]:
model_6 = FourLayerRegressionDropout().to(device)

In [ ]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model_6.parameters(), lr=0.001)

In [ ]:
train_val(model_6, 10, criterion, optimizer, train_hh_loader, val_hh_loader, transformer=transformer)

Epoch 1/10
Mean Train Loss: 0.480264
Val Loss: 0.428235, R²: 0.5483, RMSE: 27858.12 тыс. руб.
Диапазон предсказаний: 3610.16 - 323712.28 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 2/10
Mean Train Loss: 0.416472
Val Loss: 0.403013, R²: 0.5781, RMSE: 26924.14 тыс. руб.
Диапазон предсказаний: 1049.08 - 451820.22 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 3/10
Mean Train Loss: 0.377639
Val Loss: 0.422518, R²: 0.5968, RMSE: 26320.45 тыс. руб.
Диапазон предсказаний: 1182.85 - 383690.28 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 4/10
Mean Train Loss: 0.340021
Val Loss: 0.415131, R²: 0.5586, RMSE: 27540.52 тыс. руб.
Диапазон предсказаний: 2034.07 - 571976.56 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
---------------------------

(0.4266470353440283, 0.5599769353866577, np.float32(27496.254))

## Обучение только на данных колонки description

In [15]:
data_tfidf = pd.DataFrame(data_tfidf.toarray())

In [16]:
class RobustDataFrameTransformer:
    def __init__(self, features_df, targets_series, clip_percentile=99):
        """
        features_df: DataFrame с фичами
        targets_series: Series с целевой переменной
        clip_percentile: процентиль для обрезки выбросов
        """
        # Сохраняем САМЫЕ ЧАСТОТНЫЕ ЗНАЧЕНИЯ (МОДЫ) для каждого признака
        self.features_mode_values = {}
        for col in features_df.columns:
            mode_values = features_df[col].mode()  # Находим самые частотные значения
            self.features_mode_values[col] = mode_values[0] if not mode_values.empty else features_df[col].median()

        # Сохраняем САМОЕ ЧАСТОТНОЕ ЗНАЧЕНИЕ для target
        target_mode = targets_series.mode()
        self.target_mode_value = target_mode[0] if not target_mode.empty else targets_series.median()

        # ЗАПОЛНЯЕМ NaN САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ
        features_filled, targets_filled = self._fill_na_with_mode(features_df, targets_series)

        # Конвертируем в numpy
        features_array = features_filled.values.astype(np.float32)
        targets_array = targets_filled.values.astype(np.float32)

        # ОБРЕЗАЕМ ВЫБРОСЫ перед вычислением статистик
        self.features_clip_value = np.percentile(np.abs(features_array), clip_percentile, axis=0)
        self.targets_clip_value = np.percentile(np.abs(targets_array), clip_percentile)

        features_clipped = np.clip(features_array, -self.features_clip_value, self.features_clip_value)
        targets_clipped = np.clip(targets_array, -self.targets_clip_value, self.targets_clip_value)

        # Вычисляем статистики на ОБРЕЗАННЫХ данных
        self.features_mean = np.mean(features_clipped, axis=0)
        self.features_std = np.std(features_clipped, axis=0)
        self.targets_mean = np.mean(targets_clipped)
        self.targets_std = np.std(targets_clipped)

        # Защита от деления на 0
        self.features_std = np.where(self.features_std < 1e-8, 1.0, self.features_std)
        if self.targets_std < 1e-8:
            self.targets_std = 1.0

        self.feature_columns = features_df.columns.tolist()
        self.clip_percentile = clip_percentile

        print(f"RobustDataFrameTransformer создан (clip {clip_percentile}%):")
        print(f"  Features: {len(self.feature_columns)} columns")
        print(f"  Targets: mean={self.targets_mean:.2f}, std={self.targets_std:.2f}")
        print(f"  Clip values - features: {np.max(self.features_clip_value):.2f}, targets: {self.targets_clip_value:.2f}")
        print(f"  NaN strategy: ЗАПОЛНЕНИЕ САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ")

    def _fill_na_with_mode(self, features_df, targets_series):
        """ЗАПОЛНЯЕТ NaN САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ"""

        # Проверяем наличие NaN
        features_nan_count = features_df.isna().sum().sum()
        targets_nan_count = targets_series.isna().sum()

        if features_nan_count > 0:
            print(f"Заполняем {features_nan_count} NaN в features САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ")
        if targets_nan_count > 0:
            print(f"Заполняем {targets_nan_count} NaN в targets САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ")

        # ЗАПОЛНЯЕМ FEATURES САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ
        features_filled = features_df.copy()
        for col, mode_value in self.features_mode_values.items():
            features_filled[col] = features_filled[col].fillna(mode_value)

        # ЗАПОЛНЯЕМ TARGET САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ
        targets_filled = targets_series.fillna(self.target_mode_value)

        return features_filled, targets_filled

    def transform_features(self, features_df):
        """Нормализует DataFrame с обрезкой выбросов и ЗАПОЛНЕНИЕМ NaN САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ"""
        # ЗАПОЛНЯЕМ NaN САМЫМИ ЧАСТОТНЫМИ ЗНАЧЕНИЯМИ
        features_filled = features_df.copy()
        for col, mode_value in self.features_mode_values.items():
            features_filled[col] = features_filled[col].fillna(mode_value)

        features_np = features_filled.values.astype(np.float32)
        features_clipped = np.clip(features_np, -self.features_clip_value, self.features_clip_value)
        features_norm = (features_clipped - self.features_mean) / self.features_std
        return features_norm

    def transform_targets(self, targets_series):
        """Нормализует Series с обрезкой выбросов и ЗАПОЛНЕНИЕМ NaN САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ"""
        # ЗАПОЛНЯЕМ NaN САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ
        targets_filled = targets_series.fillna(self.target_mode_value)

        targets_np = targets_filled.values.astype(np.float32)
        targets_clipped = np.clip(targets_np, -self.targets_clip_value, self.targets_clip_value)
        targets_norm = (targets_clipped - self.targets_mean) / self.targets_std
        return targets_norm

    def inverse_transform_targets(self, targets_norm):
        """Денормализует предсказания"""
        if isinstance(targets_norm, torch.Tensor):
            targets_norm = targets_norm.detach().cpu().numpy()
        return targets_norm * self.targets_std + self.targets_mean

In [17]:
print("=== ПОДГОТОВКА ДАННЫХ ИЗ DATAFRAME ===")

# Шаг 1: Проверяем данные
print("Исходные данные:")
print(f"Features shape: {data_tfidf.shape}")
print(f"Targets shape: {y_log.shape}")

print("\n=== СТАТИСТИКА ДО НОРМАЛИЗАЦИИ ===")
print("Features статистика:")
print(f"  Среднее: {data_tfidf.mean().mean():.3f}")
print(f"  Std: {data_tfidf.std().mean():.3f}")
print(f"  Min: {data_tfidf.min().min():.3f}")
print(f"  Max: {data_tfidf.max().max():.3f}")

print("Targets статистика:")
print(f"  Среднее: {y_log.mean():.3f}")
print(f"  Std: {y_log.std():.3f}")
print(f"  Min: {y_log.min():.3f}")
print(f"  Max: {y_log.max():.3f}")

# Шаг 2: Создаем трансформер на ВСЕХ данных
print("\n=== СОЗДАНИЕ ТРАНСФОРМЕРА ===")
transformer = RobustDataFrameTransformer(data_tfidf, y_log)

# Шаг 3: Нормализуем данные
print("\n=== НОРМАЛИЗАЦИЯ ДАННЫХ ===")
normalized_features = transformer.transform_features(data_tfidf)
normalized_targets = transformer.transform_targets(y_log)

print(f"Нормализованные features: {normalized_features.min():.3f} to {normalized_features.max():.3f}")
print(f"Нормализованные targets: {normalized_targets.min():.3f} to {normalized_targets.max():.3f}")

# Шаг 4: Преобразуем в тензоры и создаем DataLoader
print("\n=== СОЗДАНИЕ DATALOADER ===")
# Конвертируем в тензоры PyTorch
features_tensor = torch.FloatTensor(normalized_features)
targets_tensor = torch.FloatTensor(normalized_targets)

# Создаем Dataset
dataset = TensorDataset(features_tensor, targets_tensor)

# Разделяем на train/val
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

# Создаем DataLoader
batch_size = 64
train_hh_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_hh_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print("✅ DataLoader созданы из DataFrame!")

=== ПОДГОТОВКА ДАННЫХ ИЗ DATAFRAME ===
Исходные данные:
Features shape: (165825, 1000)
Targets shape: (165825,)

=== СТАТИСТИКА ДО НОРМАЛИЗАЦИИ ===
Features статистика:
  Среднее: 0.007
  Std: 0.029
  Min: 0.000
  Max: 1.000
Targets статистика:
  Среднее: 10.950
  Std: 0.593
  Min: 6.909
  Max: 14.221

=== СОЗДАНИЕ ТРАНСФОРМЕРА ===
RobustDataFrameTransformer создан (clip 99%):
  Features: 1000 columns
  Targets: mean=10.95, std=0.59
  Clip values - features: 0.41, targets: 12.43
  NaN strategy: ЗАПОЛНЕНИЕ САМЫМ ЧАСТОТНЫМ ЗНАЧЕНИЕМ

=== НОРМАЛИЗАЦИЯ ДАННЫХ ===
Нормализованные features: -1.323 to 9.936
Нормализованные targets: -6.885 to 2.525

=== СОЗДАНИЕ DATALOADER ===
Train samples: 132660
Val samples: 33165
✅ DataLoader созданы из DataFrame!


In [29]:
model_7 = SimpleRegression().to(device)

In [30]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model_7.parameters(), lr=0.001)

In [28]:
import torch.nn as nn

class SimpleRegression(nn.Module):
    def __init__(self, input_size=1000):
        super().__init__()
        # Всего 1 выходной нейрон для предсказания одного числа
        self.linear = nn.Linear(input_size, 1)

    def forward(self, x):
        x = self.linear(x)

        return x.squeeze(1)

In [31]:
train_val(model_7, 10, criterion, optimizer, train_hh_loader, val_hh_loader, transformer=transformer)

Epoch 1/10
Mean Train Loss: 0.600612
Val Loss: 0.603221, R²: 0.3832, RMSE: 32622.64 тыс. руб.
Диапазон предсказаний: 13944.54 - 309065.47 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 2/10
Mean Train Loss: 0.587089
Val Loss: 0.594227, R²: 0.3903, RMSE: 32435.01 тыс. руб.
Диапазон предсказаний: 12332.41 - 274847.00 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 3/10
Mean Train Loss: 0.587240
Val Loss: 0.589770, R²: 0.3819, RMSE: 32656.75 тыс. руб.
Диапазон предсказаний: 14494.69 - 325869.03 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 4/10
Mean Train Loss: 0.586866
Val Loss: 0.601004, R²: 0.3779, RMSE: 32763.22 тыс. руб.
Диапазон предсказаний: 13821.38 - 314225.81 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
-----------------------

(0.5972378320546968, 0.3731899857521057, np.float32(32887.04))

In [32]:
class TwoLayerRegression(nn.Module):
    def __init__(self, input_size=1000, hidden_size=500):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, 1)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)

        return x.squeeze(1)

In [33]:
model_8 = TwoLayerRegression().to(device)

In [34]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model_8.parameters(), lr=0.001)

In [35]:
train_val(model_8, 10, criterion, optimizer, train_hh_loader, val_hh_loader, transformer=transformer)

Epoch 1/10
Mean Train Loss: 0.539529
Val Loss: 0.511741, R²: 0.4645, RMSE: 30397.45 тыс. руб.
Диапазон предсказаний: 5985.79 - 317519.84 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 2/10
Mean Train Loss: 0.470945
Val Loss: 0.507010, R²: 0.4776, RMSE: 30023.01 тыс. руб.
Диапазон предсказаний: 2980.61 - 380505.44 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 3/10
Mean Train Loss: 0.411973
Val Loss: 0.497297, R²: 0.5001, RMSE: 29368.23 тыс. руб.
Диапазон предсказаний: 3306.79 - 417303.00 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 4/10
Mean Train Loss: 0.348228
Val Loss: 0.483453, R²: 0.5150, RMSE: 28929.60 тыс. руб.
Диапазон предсказаний: 1891.75 - 382043.91 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
---------------------------

(0.5265880897210512, 0.48848336935043335, np.float32(29708.908))

In [37]:
class TwoLayerRegressionDropout(nn.Module):
    def __init__(self, input_size=1000, hidden_size=500, dropout_rate=0.3):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, 1)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)

        return x.squeeze(1)

In [38]:
model_9 = TwoLayerRegressionDropout().to(device)

In [39]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model_9.parameters(), lr=0.001)

In [40]:
train_val(model_9, 10, criterion, optimizer, train_hh_loader, val_hh_loader, transformer=transformer)

Epoch 1/10
Mean Train Loss: 0.554752
Val Loss: 0.541348, R²: 0.4382, RMSE: 31135.76 тыс. руб.
Диапазон предсказаний: 5990.09 - 318178.84 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 2/10
Mean Train Loss: 0.501774
Val Loss: 0.501310, R²: 0.4855, RMSE: 29795.37 тыс. руб.
Диапазон предсказаний: 3263.03 - 324014.34 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 3/10
Mean Train Loss: 0.466427
Val Loss: 0.499615, R²: 0.4753, RMSE: 30088.70 тыс. руб.
Диапазон предсказаний: 3995.18 - 291101.41 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 4/10
Mean Train Loss: 0.430440
Val Loss: 0.479716, R²: 0.5079, RMSE: 29140.96 тыс. руб.
Диапазон предсказаний: 2052.10 - 431566.69 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
---------------------------

(0.48340180079831324, 0.5210054516792297, np.float32(28748.959))

In [42]:
class FourLayerRegressionDropout(nn.Module):
    def __init__(self, input_size=1000, hidden_size1=512,  hidden_size2=256,  hidden_size3=64, dropout_rate=0.3):
        super().__init__()
        self.linear1 = nn.Linear(input_size, hidden_size1)
        self.linear2 = nn.Linear(hidden_size1, hidden_size2)
        self.linear3 = nn.Linear(hidden_size2, hidden_size3)
        self.linear4 = nn.Linear(hidden_size3, 1)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear3(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear4(x)

        return x.squeeze(1)

In [43]:
model_10 = FourLayerRegressionDropout().to(device)

In [44]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model_10.parameters(), lr=0.001)

In [45]:
train_val(model_10, 10, criterion, optimizer, train_hh_loader, val_hh_loader, transformer=transformer)

Epoch 1/10
Mean Train Loss: 0.550508
Val Loss: 0.529257, R²: 0.4832, RMSE: 29860.69 тыс. руб.
Диапазон предсказаний: 6743.55 - 299279.22 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 2/10
Mean Train Loss: 0.484022
Val Loss: 0.472558, R²: 0.5116, RMSE: 29028.91 тыс. руб.
Диапазон предсказаний: 1104.07 - 325872.16 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 3/10
Mean Train Loss: 0.444817
Val Loss: 0.470165, R²: 0.5204, RMSE: 28765.84 тыс. руб.
Диапазон предсказаний: 521.30 - 371243.62 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
--------------------------------------------------
Epoch 4/10
Mean Train Loss: 0.408245
Val Loss: 0.458637, R²: 0.5389, RMSE: 28205.50 тыс. руб.
Диапазон предсказаний: 1197.16 - 523761.53 тыс. руб.
Диапазон истинных значений: 1000.00 - 250000.00 тыс. руб.
----------------------------

(0.4536491135756175, 0.5393024682998657, np.float32(28194.523))